# PD LEDD Change — Conformal Prediction Comparison

Compares five conformal prediction methods on real PD data across α ∈ {0.80, 0.85, 0.90, 0.95}.

| Method | Label | Description |
|--------|-------|-------------|
| `adapted_original_pipeline` | **CPCI** | Proposed two-stage method|
| `adapted_vanilla_model` | **VCI** | Vanilla conformal baseline |
| `adapted_RF_XGB_model` | **CPCI-XGB** | CPCI with RF classifier + XGBoost regressor |
| `adapted_ad_hoc_model` | **WEIGHTED-VCI** | Unified weighted nonconformity score |
| `adapted_class_conditional_model` | **CLASS-COND** | Separate calibration per class |


## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import MinMaxScaler
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import roc_auc_score
from venn_abers              import VennAbers

RANDOM_STATE   = 42
LARGE_CONSTANT = 100_000_000
ALPHA_VALUES   = [0.80, 0.85, 0.90, 0.95]


## 2 · Load and preprocess data

In [ ]:
data = pd.read_csv('data/data_1Y.csv')

# Regression target — save before dropping
normalized_pctg_change = data['normalized_percent_change']
data.drop(columns=['normalized_percent_change'], inplace=True)

# One-hot encode demographics
demographic_vars = ['gender_source_value', 'race_source_value', 'ethnicity_source_value']
xgboost_df = pd.get_dummies(data, columns=demographic_vars)

# MinMaxScale numeric features
scaler = MinMaxScaler()
numeric_vars = ['mean_led_per_visit', 'age', 'length_of_stay',
                'days_since_last_visit', 'days_to_diagnosis']
for var in numeric_vars:
    if var in xgboost_df.columns:
        xgboost_df[var] = scaler.fit_transform(xgboost_df[[var]])

# Keep 'prediction' last
pred_col = xgboost_df.pop('prediction')
xgboost_df['prediction'] = pred_col

X        = xgboost_df.iloc[:, :-1].values
y_binary = xgboost_df.iloc[:,  -1].values
y_cont   = normalized_pctg_change.values

print(f"X shape      : {X.shape}")
print(f"y_binary     : {np.bincount(y_binary.astype(int))} (0=no-change, 1=change)")
print(f"y_cont range : [{y_cont.min():.3f}, {y_cont.max():.3f}]")
print(f"Zero fraction: {(y_cont == 0).mean():.1%}")


## 3 · Model parameters

In [ ]:
binary_params = {
    'max_depth':       7,
    'learning_rate':   0.01,
    'n_estimators':    800,
    'objective':       'binary:logistic',
    'eval_metric':     'auc',
    'use_label_encoder': False,
    'verbosity':       0,
    'random_state':    RANDOM_STATE
}

regression_params_short = {
    'max_depth':     7,
    'learning_rate': 0.1,
    'n_estimators':  700,
    'objective':     'reg:squarederror',
    'verbosity':     0,
    'random_state':  RANDOM_STATE
}


## 4 · Data splitting helper

In [ ]:
def split_into_five_equal_parts(X, y_binary, y_continuous, random_state=42):
    """
    Splits data into 5 equal parts (20% each):
    train | calib1 | calib2 | calib3 | test
    """
    X_temp, X_test, yb_temp, yb_test, yc_temp, yc_test = train_test_split(
        X, y_binary, y_continuous, test_size=0.2, random_state=random_state)

    X_train, X_temp2, yb_train, yb_temp2, yc_train, yc_temp2 = train_test_split(
        X_temp, yb_temp, yc_temp, test_size=0.75, random_state=random_state)

    X_c1, X_temp3, yb_c1, yb_temp3, yc_c1, yc_temp3 = train_test_split(
        X_temp2, yb_temp2, yc_temp2, test_size=0.5, random_state=random_state)

    X_c2, X_c3, yb_c2, yb_c3, yc_c2, yc_c3 = train_test_split(
        X_temp3, yb_temp3, yc_temp3, test_size=0.5, random_state=random_state)

    return (X_train, X_c1, X_c2, X_c3, X_test,
            yb_train, yb_c1, yb_c2, yb_c3, yb_test,
            yc_train, yc_c1, yc_c2, yc_c3, yc_test)


## 5 · Method 1 — CPCI

In [ ]:
def adapted_original_pipeline(X, y_binary, y_continuous,
                               alpha_tilda=0.9, choose_method="original",
                               random_state=RANDOM_STATE):
    """
    Proposed CPCI method: XGBoost classifier + XGBoost regressor.
    Three calibration sets. Tunes threshold r to minimise interval length.
    """
    np.random.seed(random_state)
    (X_train, X_c1, X_c2, X_c3, X_test,
     yb_train, yb_c1, yb_c2, yb_c3, yb_test,
     yc_train, yc_c1, yc_c2, yc_c3, yc_test
     ) = split_into_five_equal_parts(X, y_binary, y_continuous, random_state)

    classifier = xgb.XGBClassifier(**binary_params)
    classifier.fit(X_train, yb_train)

    nz = yb_train == 1
    reg_nz = xgb.XGBRegressor(**regression_params_short)
    reg_nz.fit(X_train[nz], yc_train[nz])

    reg_all = xgb.XGBRegressor(**regression_params_short)
    reg_all.fit(X_train, yc_train)

    p1 = classifier.predict_proba(X_c1)[:, 1]
    p2 = classifier.predict_proba(X_c2)[:, 1]
    p3 = classifier.predict_proba(X_c3)[:, 1]

    results = []
    for r in np.arange(0.1, 0.95, 0.05):
        alpha_r = np.quantile(np.append(p1, LARGE_CONSTANT), r)

        zeros2 = p2 <= alpha_r
        if not np.any(zeros2): continue

        beta_hat = np.mean(yb_c2[zeros2] == 0)
        n2       = len(X_c2)
        beta_adj = beta_hat - 0.1 * np.sqrt(np.log(n2) / n2)
        if np.isnan(beta_adj): continue

        cur = {'r': r, 'alpha_r': alpha_r, 'fallback': False, 'length': np.inf}

        if beta_adj < alpha_tilda:
            resid          = np.abs(yc_c3 - reg_all.predict(X_c3))
            cur['length']  = np.quantile(np.append(resid, LARGE_CONSTANT), alpha_tilda)
            cur['fallback'] = True
        else:
            fq = (alpha_tilda + 1/len(X_c3) - beta_adj * r) / (1 - r)
            if fq > 1: continue
            fq = max(0, fq)
            nz3 = p3 > alpha_r
            if not np.any(nz3): continue
            resid     = np.abs(yc_c3[nz3] - reg_nz.predict(X_c3[nz3]))
            raw       = np.quantile(np.append(resid, LARGE_CONSTANT), fq)
            cur['length'] = raw * (1 - r) if choose_method == "original" else raw

        results.append(cur)

    if not results:
        raise ValueError("No valid r found.")

    best = min(results, key=lambda x: x['length'])
    lb = np.zeros(len(X_test))
    ub = np.zeros(len(X_test))

    if best['fallback']:
        preds = reg_all.predict(X_test)
        lb = preds - best['length']
        ub = preds + best['length']
    else:
        pt    = classifier.predict_proba(X_test)[:, 1]
        nz_t  = pt > best['alpha_r']
        preds = reg_nz.predict(X_test)
        m     = best['length'] / (1 - best['r']) if choose_method == "original" else best['length']
        lb[nz_t] = preds[nz_t] - m
        ub[nz_t] = preds[nz_t] + m

    w   = (yc_test >= lb) & (yc_test <= ub)
    nz  = yb_test == 1
    lens = ub - lb
    return lb, ub, yc_test, {
        'coverage':          np.mean(w),
        'coverage_nonzero':  np.mean(w[nz]) if np.any(nz) else np.nan,
        'avg_width':         np.mean(lens),
        'avg_width_nonzero': np.mean(lens[nz]) if np.any(nz) else np.nan,
        'method_used':       'VCI' if best['fallback'] else 'CPCI',
        'best_r':            best['r']
    }


## 6 · Method 2 — VCI (Vanilla)

In [ ]:
def adapted_vanilla_model(X, y_binary, y_continuous,
                           alpha_tilda=0.9, random_state=RANDOM_STATE):
    """VCI: single XGBoost regressor on all data, one combined calibration set."""
    np.random.seed(random_state)
    (X_train, X_c1, X_c2, X_c3, X_test,
     yb_train, yb_c1, yb_c2, yb_c3, yb_test,
     yc_train, yc_c1, yc_c2, yc_c3, yc_test
     ) = split_into_five_equal_parts(X, y_binary, y_continuous, random_state)

    reg = xgb.XGBRegressor(**regression_params_short)
    reg.fit(X_train, yc_train)

    X_cal = np.vstack([X_c1, X_c2, X_c3])
    y_cal = np.concatenate([yc_c1, yc_c2, yc_c3])
    resid = np.abs(y_cal - reg.predict(X_cal))
    hw    = np.quantile(np.append(resid, LARGE_CONSTANT), alpha_tilda)

    preds = reg.predict(X_test)
    lb, ub = preds - hw, preds + hw

    w   = (yc_test >= lb) & (yc_test <= ub)
    nz  = yb_test == 1
    lens = ub - lb
    return lb, ub, yc_test, {
        'coverage':          np.mean(w),
        'coverage_nonzero':  np.mean(w[nz]) if np.any(nz) else np.nan,
        'avg_width':         np.mean(lens),
        'avg_width_nonzero': np.mean(lens[nz]) if np.any(nz) else np.nan,
        'method_used':       'VCI'
    }


## 7 · Method 3 — CPCI-XGB (RF classifier)

In [ ]:
def adapted_RF_XGB_model(X, y_binary, y_continuous,
                          alpha_tilda=0.9, choose_method="original",
                          random_state=RANDOM_STATE):
    """CPCI-XGB: Random Forest classifier + XGBoost regressor."""
    np.random.seed(random_state)
    (X_train, X_c1, X_c2, X_c3, X_test,
     yb_train, yb_c1, yb_c2, yb_c3, yb_test,
     yc_train, yc_c1, yc_c2, yc_c3, yc_test
     ) = split_into_five_equal_parts(X, y_binary, y_continuous, random_state)

    clf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                  random_state=random_state)
    clf.fit(X_train, yb_train)

    nz = yb_train == 1
    reg_nz  = xgb.XGBRegressor(**regression_params_short)
    reg_nz.fit(X_train[nz], yc_train[nz])
    reg_all = xgb.XGBRegressor(**regression_params_short)
    reg_all.fit(X_train, yc_train)

    p1 = clf.predict_proba(X_c1)[:, 1]
    p2 = clf.predict_proba(X_c2)[:, 1]
    p3 = clf.predict_proba(X_c3)[:, 1]

    results = []
    for r in np.arange(0.1, 0.95, 0.05):
        alpha_r = np.quantile(np.append(p1, LARGE_CONSTANT), r)
        zeros2  = p2 <= alpha_r
        if not np.any(zeros2): continue
        beta_hat = np.mean(yb_c2[zeros2] == 0)
        beta_adj = beta_hat - 0.1 * np.sqrt(np.log(len(X_c2)) / len(X_c2))
        if np.isnan(beta_adj): continue

        cur = {'r': r, 'alpha_r': alpha_r, 'fallback': False, 'length': np.inf}
        if beta_adj < alpha_tilda:
            resid = np.abs(yc_c3 - reg_all.predict(X_c3))
            cur['length']   = np.quantile(np.append(resid, LARGE_CONSTANT), alpha_tilda)
            cur['fallback'] = True
        else:
            fq = (alpha_tilda + 1/len(X_c3) - beta_adj * r) / (1 - r)
            if fq > 1: continue
            fq = max(0, fq)
            nz3 = p3 > alpha_r
            if not np.any(nz3): continue
            resid = np.abs(yc_c3[nz3] - reg_nz.predict(X_c3[nz3]))
            raw   = np.quantile(np.append(resid, LARGE_CONSTANT), fq)
            cur['length'] = raw * (1 - r) if choose_method == "original" else raw
        results.append(cur)

    if not results: raise ValueError("No valid r found.")
    best = min(results, key=lambda x: x['length'])
    lb = np.zeros(len(X_test)); ub = np.zeros(len(X_test))

    if best['fallback']:
        preds = reg_all.predict(X_test)
        lb, ub = preds - best['length'], preds + best['length']
    else:
        pt   = clf.predict_proba(X_test)[:, 1]
        nz_t = pt > best['alpha_r']
        preds = reg_nz.predict(X_test)
        m = best['length'] / (1-best['r']) if choose_method=="original" else best['length']
        lb[nz_t] = preds[nz_t] - m
        ub[nz_t] = preds[nz_t] + m

    w = (yc_test >= lb) & (yc_test <= ub)
    nz = yb_test == 1; lens = ub - lb
    return lb, ub, yc_test, {
        'coverage':          np.mean(w),
        'coverage_nonzero':  np.mean(w[nz]) if np.any(nz) else np.nan,
        'avg_width':         np.mean(lens),
        'avg_width_nonzero': np.mean(lens[nz]) if np.any(nz) else np.nan,
        'method_used':       'VCI' if best['fallback'] else 'CPCI-XGB',
        'best_r':            best['r']
    }


## 8 · Method 4 — WEIGHTED-VCI

In [ ]:
def adapted_ad_hoc_model(X, y_binary, y_continuous,
                          alpha_tilda=0.9, random_state=RANDOM_STATE):
    """
    WEIGHTED-VCI: unified weighted nonconformity score.
    Score: y=0 → 0.5*P(Y≠0|x)  |  y≠0 → 0.5*|y-ŷ|
    Q = quantile(scores, alpha); radius = 2Q
    Zero included if P(Y≠0) ≤ radius.
    """
    np.random.seed(random_state)
    (X_train, X_c1, X_c2, X_c3, X_test,
     yb_train, yb_c1, yb_c2, yb_c3, yb_test,
     yc_train, yc_c1, yc_c2, yc_c3, yc_test
     ) = split_into_five_equal_parts(X, y_binary, y_continuous, random_state)

    X_cal = np.vstack([X_c1, X_c2, X_c3])
    y_cal = np.concatenate([yc_c1, yc_c2, yc_c3])

    reg = xgb.XGBRegressor(**regression_params_short)
    reg.fit(X_train, yc_train)
    clf = xgb.XGBClassifier(**binary_params)
    clf.fit(X_train, yb_train)

    yhat = reg.predict(X_cal)
    p0   = clf.predict_proba(X_cal)[:, 0]
    scores = np.where(y_cal == 0,
                      0.5 * (1 - p0),
                      0.5 * np.abs(y_cal - yhat))

    Q      = np.quantile(np.append(scores, LARGE_CONSTANT), alpha_tilda)
    radius = 2 * Q

    yhat_t = reg.predict(X_test)
    p0_t   = clf.predict_proba(X_test)[:, 0]
    lb = yhat_t - radius
    ub = yhat_t + radius
    incl_zero = (1 - p0_t) <= radius
    zero_cont = (lb <= 0) & (ub >= 0)

    cov_cont  = (yc_test >= lb) & (yc_test <= ub)
    cov_zero  = (yc_test == 0) & incl_zero
    covered   = cov_cont | cov_zero

    nz = yb_test == 1; lens = ub - lb
    return lb, ub, yc_test, {
        'coverage':          np.mean(covered),
        'coverage_nonzero':  np.mean(covered[nz]) if np.any(nz) else np.nan,
        'avg_width':         np.mean(lens),
        'avg_width_nonzero': np.mean(lens[nz]) if np.any(nz) else np.nan,
        'method_used':       'WEIGHTED-VCI',
        'Q': Q, 'radius': radius,
        'inclusion_zero': np.mean(zero_cont | incl_zero),
        'n_disjoint':     int(np.sum(incl_zero & ~zero_cont))
    }


## 9 · Method 5 — CLASS-COND

In [ ]:
def adapted_class_conditional_model(X, y_binary, y_continuous,
                                     alpha_tilda=0.9, random_state=RANDOM_STATE):
    """
    CLASS-COND: separate class calibration.
    Regression calib on non-zeros → interval width.
    Classification calib on zeros → prob threshold for disjoint {0}.
    """
    np.random.seed(random_state)
    (X_train, X_c1, X_c2, X_c3, X_test,
     yb_train, yb_c1, yb_c2, yb_c3, yb_test,
     yc_train, yc_c1, yc_c2, yc_c3, yc_test
     ) = split_into_five_equal_parts(X, y_binary, y_continuous, random_state)

    X_cal  = np.vstack([X_c1, X_c2, X_c3])
    yc_cal = np.concatenate([yc_c1, yc_c2, yc_c3])
    yb_cal = np.concatenate([yb_c1, yb_c2, yb_c3])

    nz = yb_train == 1
    reg = xgb.XGBRegressor(**regression_params_short)
    reg.fit(X_train[nz], yc_train[nz])
    clf = xgb.XGBClassifier(**binary_params)
    clf.fit(X_train, yb_train)

    # Regression calibration (non-zero only)
    resid    = np.abs(yc_cal - reg.predict(X_cal))
    resid_nz = resid[yb_cal == 1]
    n_nz     = len(resid_nz)
    reg_q    = min(((n_nz + 1) * alpha_tilda) / n_nz, 1.0)
    iw       = np.quantile(np.append(resid_nz, LARGE_CONSTANT), reg_q)

    # Classification calibration (zero only)
    probs_cal  = clf.predict_proba(X_cal)[:, 1]
    probs_zero = probs_cal[yb_cal == 0]
    n_z        = len(probs_zero)
    cls_q      = min(((n_z + 1) * alpha_tilda) / n_z, 1.0)
    thr        = np.quantile(np.append(probs_zero, LARGE_CONSTANT), cls_q)

    preds_t = reg.predict(X_test)
    probs_t = clf.predict_proba(X_test)[:, 1]
    lb = preds_t - iw
    ub = preds_t + iw

    zero_cont = (lb <= 0) & (ub >= 0)
    add_zero  = (probs_t < thr) & (~zero_cont)
    cov_cont  = (yc_test >= lb) & (yc_test <= ub)
    cov_zero  = (yc_test == 0) & add_zero
    covered   = cov_cont | cov_zero

    nz = yb_test == 1; lens = ub - lb
    return lb, ub, yc_test, {
        'coverage':          np.mean(covered),
        'coverage_nonzero':  np.mean(covered[nz]) if np.any(nz) else np.nan,
        'avg_width':         np.mean(lens),
        'avg_width_nonzero': np.mean(lens[nz]) if np.any(nz) else np.nan,
        'method_used':       'CLASS-COND',
        'interval_width':    iw,
        'prob_threshold':    thr,
        'inclusion_zero':    np.mean(zero_cont | add_zero),
        'n_disjoint':        int(np.sum(add_zero))
    }


## 10 · Run all methods across α values

In [ ]:
methods = {
    'CPCI':          lambda a: adapted_original_pipeline(X, y_binary, y_cont, alpha_tilda=a),
    'VCI':           lambda a: adapted_vanilla_model(X, y_binary, y_cont, alpha_tilda=a),
    'CPCI-XGB':      lambda a: adapted_RF_XGB_model(X, y_binary, y_cont, alpha_tilda=a),
    'WEIGHTED-VCI':  lambda a: adapted_ad_hoc_model(X, y_binary, y_cont, alpha_tilda=a),
    'CLASS-COND':    lambda a: adapted_class_conditional_model(X, y_binary, y_cont, alpha_tilda=a),
}

results = {name: {
    'alpha':                [],
    'coverage':             [],
    'coverage_nonzero':     [],
    'avg_width':            [],
    'avg_width_nonzero':    [],
    'method_used':          []
} for name in methods}

for alpha in ALPHA_VALUES:
    print(f"\nα = {alpha}")
    for name, fn in methods.items():
        try:
            _, _, _, m = fn(alpha)
            results[name]['alpha'].append(alpha)
            results[name]['coverage'].append(m['coverage'])
            results[name]['coverage_nonzero'].append(m['coverage_nonzero'])
            results[name]['avg_width'].append(m['avg_width'])
            results[name]['avg_width_nonzero'].append(m['avg_width_nonzero'])
            results[name]['method_used'].append(m.get('method_used', name))
            print(f"  {name:<16} cov={m['coverage']:.3f}  cov_nz={m['coverage_nonzero']:.3f}"
                  f"  width={m['avg_width']:.4f}  width_nz={m['avg_width_nonzero']:.4f}"
                  f"  [{m.get('method_used', '')}]")
        except Exception as e:
            print(f"  {name:<16} ERROR: {e}")


## 11 · Summary table

In [ ]:
rows = []
for name, r in results.items():
    for i, alpha in enumerate(r['alpha']):
        rows.append({
            'Method':            name,
            'Alpha':             alpha,
            'Coverage':          f"{r['coverage'][i]:.3f}",
            'Cov (non-zero)':    f"{r['coverage_nonzero'][i]:.3f}",
            'Width (overall)':   f"{r['avg_width'][i]:.4f}",
            'Width (non-zero)':  f"{r['avg_width_nonzero'][i]:.4f}",
        })

summary = pd.DataFrame(rows)
summary_pivot = summary.pivot(index='Method', columns='Alpha')
print(summary_pivot.to_string())
summary.to_csv('conformal_comparison_results.csv', index=False)
print("\nSaved to conformal_comparison_results.csv")


## 12 · Plots

In [ ]:
# Color / style scheme — one entry per method
style = {
    'CPCI':         {'color': 'green',   'marker': 'o', 'ls': '-'},
    'VCI':          {'color': 'red',     'marker': 'o', 'ls': '--'},
    'CPCI-XGB':     {'color': 'purple',  'marker': 's', 'ls': '-'},
    'WEIGHTED-VCI': {'color': 'orange',  'marker': 'D', 'ls': '-'},
    'CLASS-COND':   {'color': 'steelblue','marker': '^', 'ls': '-'},
}

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
x_ticks = ALPHA_VALUES

# ── Coverage ─────────────────────────────────────────────────────────────────
for name, r in results.items():
    s = style[name]
    ax[0].plot(r['alpha'], r['coverage'],
               color=s['color'], marker=s['marker'], linestyle=s['ls'],
               label=f"{name}")
    ax[0].plot(r['alpha'], r['coverage_nonzero'],
               color=s['color'], marker=s['marker'], linestyle=':',
               alpha=0.55)

# Identity line (desired coverage)
ax[0].plot(x_ticks, x_ticks, linestyle='--', color='gray',
           linewidth=1, label='Desired (α)')
ax[0].set_xticks(x_ticks); ax[0].set_xticklabels(x_ticks)
ax[0].set_xlabel('α', fontsize=13)
ax[0].set_ylabel('Coverage', fontsize=13)
ax[0].set_title('Coverage (solid=overall, dotted=non-zero)', fontsize=12)
ax[0].grid(True, alpha=0.4)
ax[0].text(-0.02, 1.02, 'A', transform=ax[0].transAxes,
           fontsize=15, fontweight='bold', va='top')

# ── Interval Width ────────────────────────────────────────────────────────────
for name, r in results.items():
    s = style[name]
    ax[1].plot(r['alpha'], r['avg_width'],
               color=s['color'], marker=s['marker'], linestyle=s['ls'],
               label=f"{name}")
    ax[1].plot(r['alpha'], r['avg_width_nonzero'],
               color=s['color'], marker=s['marker'], linestyle=':',
               alpha=0.55)

ax[1].set_xticks(x_ticks); ax[1].set_xticklabels(x_ticks)
ax[1].set_xlabel('α', fontsize=13)
ax[1].set_ylabel('Average Interval Width', fontsize=13)
ax[1].set_title('Interval Width (solid=overall, dotted=non-zero)', fontsize=12)
ax[1].grid(True, alpha=0.4)
ax[1].text(-0.12, 1.02, 'B', transform=ax[1].transAxes,
           fontsize=15, fontweight='bold', va='top')

handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=6,
           fontsize=10, bbox_to_anchor=(0.5, 1.0), frameon=True)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('conformal_comparison.png', transparent=True, bbox_inches='tight', dpi=150)
plt.show()
print("Saved to conformal_comparison.png")


## 13 · Coverage vs Width per method

In [ ]:
# One row per metric, matching the exact style from Zhirui's notebook output

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for name, r in results.items():
    s = style[name]
    ax[0].plot(r['alpha'], r['coverage'],
               color=s['color'], marker=s['marker'], linestyle=s['ls'], label=name)

ax[0].plot(x_ticks, x_ticks, linestyle='--', color='black',
           linewidth=1.2, label='Desired (α)')
ax[0].set_xticks(x_ticks); ax[0].set_xticklabels(x_ticks)
ax[0].set_xlabel('Alpha', fontsize=14)
ax[0].set_ylabel('Average Overall Coverage', fontsize=14)
ax[0].set_title('Coverage', fontsize=14)
ax[0].grid(True, alpha=0.6)
ax[0].text(0.0, 0.99, 'A', transform=ax[0].transAxes,
           fontsize=16, fontweight='bold', va='top', ha='right')

for name, r in results.items():
    s = style[name]
    ax[1].plot(r['alpha'], r['avg_width'],
               color=s['color'], marker=s['marker'], linestyle=s['ls'], label=name)

ax[1].set_xticks(x_ticks); ax[1].set_xticklabels(x_ticks)
ax[1].set_xlabel('Alpha', fontsize=14)
ax[1].set_ylabel('Average Overall Interval Width', fontsize=14)
ax[1].set_title('Interval Width', fontsize=14)
ax[1].grid(True, alpha=0.6)
ax[1].text(-0.120, 0.99, 'B', transform=ax[1].transAxes,
           fontsize=16, fontweight='bold', va='top', ha='right')

handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=6,
           fontsize=10, bbox_to_anchor=(0.5, 0.95), frameon=True)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.savefig('conformal_comparison_overall.png', transparent=True, bbox_inches='tight', dpi=150)
plt.show()

# Non-zero version
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for name, r in results.items():
    s = style[name]
    ax[0].plot(r['alpha'], r['coverage_nonzero'],
               color=s['color'], marker=s['marker'], linestyle=s['ls'], label=name)

ax[0].plot(x_ticks, x_ticks, linestyle='--', color='black', linewidth=1.2, label='Desired (α)')
ax[0].set_xticks(x_ticks); ax[0].set_xticklabels(x_ticks)
ax[0].set_xlabel('Alpha', fontsize=14)
ax[0].set_ylabel('Coverage (Non-Zero Patients)', fontsize=14)
ax[0].set_title('Coverage — Non-Zero Only', fontsize=14)
ax[0].grid(True, alpha=0.6)
ax[0].text(0.0, 0.99, 'A', transform=ax[0].transAxes,
           fontsize=16, fontweight='bold', va='top', ha='right')

for name, r in results.items():
    s = style[name]
    ax[1].plot(r['alpha'], r['avg_width_nonzero'],
               color=s['color'], marker=s['marker'], linestyle=s['ls'], label=name)

ax[1].set_xticks(x_ticks); ax[1].set_xticklabels(x_ticks)
ax[1].set_xlabel('Alpha', fontsize=14)
ax[1].set_ylabel('Interval Width (Non-Zero Patients)', fontsize=14)
ax[1].set_title('Interval Width — Non-Zero Only', fontsize=14)
ax[1].grid(True, alpha=0.6)
ax[1].text(-0.120, 0.99, 'B', transform=ax[1].transAxes,
           fontsize=16, fontweight='bold', va='top', ha='right')

handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=6,
           fontsize=10, bbox_to_anchor=(0.5, 0.95), frameon=True)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.savefig('conformal_comparison_nonzero.png', transparent=True, bbox_inches='tight', dpi=150)
plt.show()
